# Normalize datasets for easier usage

Execute this dataset for each one of the locations analysed in the thesis.

The resulting datasets are stored in NetCDF files, one per year, with variables following the CMIP6 names where available.

## Align coordinates

Datasets may be compared only if they have same coordinates.

Regarding spatial dimensions, ERA5 and NEX-GDDP-CMIP6 datasets have same Geographic Coordinate System (GCS), same geodetic datum and same resolution, but the coordinate points refer to the [top-left vertex of cells in ERA5](https://confluence.ecmwf.int/display/CKB/ERA5%3A+What+is+the+spatial+reference#heading-Visualisationofregularlatlondata), while they refer to the [center of cells in NEX-GDDP-CMIP6](https://www.nccs.nasa.gov/sites/default/files/NEX-GDDP-CMIP6-Tech_Note_4.pdf). This offset can be easily corrected by a translation, a full regridding is not necessary.
In particular, the NEX-GDDP-CMIP6 convention is chosen, hence spatial coordinates refer to the center of the grid point.

A similar issue occurs for the temporal dimension: coordinates in ERA5 and their timestamps point to the start of the day (i.e. T00:00:00), while in NEX-GDDP-CMIP6 timestamps are aligned with midday (i.e. T12:00:00). Since data have daily frequency, this difference does not pose problems when operations are applied to aggregated (i.e. downsampled) data, but may create confusion in pointwise calculations.
Hence a simple reindexing is applied to the temporal dimension of NEX-GDDP-CMIP6 to match ERA5 convention.

## References

Inside the cells of this notebook other more specific resources may appear or repeat the following:

- CMIP6 variables https://clipc-services.ceda.ac.uk/dreq/mipVars.html, it this site is not available, search directly in [this page](https://clipc-services.ceda.ac.uk/dreq/index/var.html);
- ERA5 documentation https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation;
- ECMWF's Parameter Database https://codes.ecmwf.int/grib/param-db/;
- CF standard names https://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html;
- Technical note on NEX-GDDP-CMIP6 available on the dataset official web page https://www.nccs.nasa.gov/services/data-collections/land-based-products/nex-gddp-cmip6;
- Strings describing the temporal frequency in every function related to time operations https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#timeseries-offset-aliases;
- Example of conversions applied to precipitation variables https://xclim.readthedocs.io/en/stable/notebooks/units.html#Smart-conversions:-Precipitation.

In [1]:
import itertools
import pathlib

import xarray
import xclim

import thesis

xarray.set_options(keep_attrs=True)

In [2]:
location = "torino"  # Select the spatial center of the analysis, cf. thesis.coords keys.

dir_data = pathlib.Path.home() / "data"
dir_data = dir_data / location / "Dataset"

dir_out = thesis.dir_data / location / "preprocessed"
dir_out.mkdir(parents=True, exist_ok=True)

## Preprocess ERA5 dataset

ERA5 dataset is provided through the Copernicus Data Store (CDS) but, due to ongoing maintenance, the service is not stable nor fast. Hence the Microsoft Planetary Computer (PC) is used to download the needed period, however data are available up to year 2020. Data for the resulting years (i.e. period 2021-2023) are downloaded from CDS.

In [3]:
dirs = dir_data / "ERA5"
ds_hourly = xarray.open_mfdataset(dirs.glob("*/*.nc"), combine_attrs="drop_conflicts", drop_variables="sfcWind")
display(ds_hourly)

<xarray.Dataset> Size: 2GB
Dimensions:    (latitude: 9, longitude: 10, time: 648672)
Coordinates:
  * latitude   (latitude) float32 36B 44.0 44.25 44.5 44.75 ... 45.5 45.75 46.0
  * longitude  (longitude) float32 40B 6.0 6.25 6.5 6.75 ... 7.5 7.75 8.0 8.25
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
Data variables:
    t2m        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 234MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    u10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Before any operation on coordinates, e.g. slicing, reorder spatial coordinates to set the correct boundaries and to [avoid issues during resampling](https://stackoverflow.com/questions/41077393/how-to-sort-the-index-of-a-xarray-dataset-dataarray).

In [4]:
ds_hourly = ds_hourly.sortby(["latitude", "longitude", "time"])
display(ds_hourly)

<xarray.Dataset> Size: 2GB
Dimensions:    (latitude: 9, longitude: 10, time: 648672)
Coordinates:
  * latitude   (latitude) float32 36B 44.0 44.25 44.5 44.75 ... 45.5 45.75 46.0
  * longitude  (longitude) float32 40B 6.0 6.25 6.5 6.75 ... 7.5 7.75 8.0 8.25
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
Data variables:
    t2m        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 234MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    u10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Align ERA5 spatial coordinates to the ones of NEX-GDDP-CMIP6.

In [5]:
ds_hourly = ds_hourly.assign_coords({
    "latitude": ds_hourly["latitude"] + thesis.spatial_resolution / 2,
    "longitude": ds_hourly["longitude"] + thesis.spatial_resolution / 2
})
display(ds_hourly)

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 648672, latitude: 9, longitude: 10)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
  * latitude   (latitude) float32 36B 44.12 44.38 44.62 ... 45.62 45.88 46.12
  * longitude  (longitude) float32 40B 6.125 6.375 6.625 ... 7.875 8.125 8.375
Data variables:
    t2m        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 234MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    u10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(744, 9, 10), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 467MB dask.array<chunksize=(271752, 9, 10), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Work on a region of 3x3 grid points around the coordinates of the location of interest.

In [6]:
offset = 1.6 * thesis.spatial_resolution  # Account for floating numbers precision.
ds_hourly = ds_hourly.sel(
    latitude=slice(thesis.coords[location][0] - offset, thesis.coords[location][0] + offset),
    longitude=slice(thesis.coords[location][1] - offset, thesis.coords[location][1] + offset),
    drop=True
)
display(ds_hourly)

<xarray.Dataset> Size: 215MB
Dimensions:    (time: 648672, latitude: 3, longitude: 3)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
  * latitude   (latitude) float32 12B 44.88 45.12 45.38
  * longitude  (longitude) float32 12B 7.375 7.625 7.875
Data variables:
    t2m        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 23MB dask.array<chunksize=(271752, 3, 3), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    u10        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 47MB dask.array<chunksize=(271752, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Due to the download procedure, `tas` variable is in data from Microsoft PC, but the same variable in ERA5 is called `t2m`, hence they merged in the same variable.

In [7]:
ds_hourly["tas"] = ds_hourly["tas"].fillna(ds_hourly["t2m"])
ds_hourly = ds_hourly.drop_vars("t2m")
display(ds_hourly)

<xarray.Dataset> Size: 192MB
Dimensions:    (time: 648672, latitude: 3, longitude: 3)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
  * latitude   (latitude) float32 12B 44.88 45.12 45.38
  * longitude  (longitude) float32 12B 7.375 7.625 7.875
Data variables:
    tas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    u10        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 47MB dask.array<chunksize=(271752, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

CMIP6 nomenclature is used for variables names and in general CF conventions are followed. Moreover, xclim needs some attributes to apply automatic conversions.
Change the dataset accordingly:

1. rename variables if needed;
2. add attributes `long_name`, `units` and `standard_name` where missing and [check the correctness](https://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html) of existing ones;
3. convert variables to the right canonical units.

Note that variable `tp` is the total precipitation in ERA5 and it has no counterpart in CMIP6. Since no other information other than the unit is given in [its database entry](https://codes.ecmwf.int/grib/param-db/228), I assume that it corresponds to the variable with CF convention standard name lwe_thickness_of_precipitation_amount.

This assumption is confirmed by comparing the values of cumulated 1 hour precipitation with the real values occured over Emilia-Romagna region in May 2023 (e.g. compare the [cumulated values over few days](https://www.cimafoundation.org/news/le-alluvioni-di-maggio-2023-una-analisi-scientifica/) over some known cities).

In [8]:
ds_hourly = ds_hourly.rename({
    "u10": "uas",
    "v10": "vas"
})

ds_hourly["latitude"].attrs.update({
    "long_name": "Latitude"
})
ds_hourly["longitude"].attrs.update({
    "long_name": "Longitude"
})
ds_hourly["time"].attrs.update({
    "long_name": "Time"
})
ds_hourly["tas"].attrs.update({
    "long_name": "Near-Surface Air Temperature",
    "units_metadata": "on-scale"
})
ds_hourly["tp"].attrs.update({
    "standard_name": "lwe_thickness_of_precipitation_amount"
})
ds_hourly["uas"].attrs.update({
    "long_name": "Eastward Near-Surface Wind"
})
ds_hourly["vas"].attrs.update({
    "long_name": "Northward Near-Surface Wind"
})

display(ds_hourly)

<xarray.Dataset> Size: 192MB
Dimensions:    (time: 648672, latitude: 3, longitude: 3)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
  * latitude   (latitude) float32 12B 44.88 45.12 45.38
  * longitude  (longitude) float32 12B 7.375 7.625 7.875
Data variables:
    tas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    uas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    vas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(271752, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Variable `sfcWind` (i.e. Near-Surface Wind Speed) is evaluated.

In [9]:
da = xclim.indicators.atmos.wind_speed_from_vector(ds=ds_hourly)[0]
# Coordinates attributes are not persisted for this indicator, copy them manually.
for c in da.coords:
    da[c].attrs = ds_hourly[c].attrs
da.attrs["long_name"] = da.attrs["long_name"].title()  # Same style of other variables.
ds_hourly = ds_hourly.assign({"sfcWind": da})
display(ds_hourly)

<xarray.Dataset> Size: 239MB
Dimensions:    (time: 648672, latitude: 3, longitude: 3)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-01 ... 2023-12-31T23:00:00
  * latitude   (latitude) float32 12B 44.88 45.12 45.38
  * longitude  (longitude) float32 12B 7.375 7.625 7.875
Data variables:
    tas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    tp         (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    uas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
    vas        (time, latitude, longitude) float64 47MB dask.array<chunksize=(271752, 3, 3), meta=np.ndarray>
    sfcWind    (time, latitude, longitude) float64 47MB dask.array<chunksize=(744, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Store hourly data in a file for record.

In [10]:
path = dir_out / "reference_hourly.nc"
ds_hourly.to_netcdf(path)

Extract daily data, pay attention to the meaning of each quantity when aggregating through time.

In [11]:
_resample = ds_hourly.resample(time="1D")
ds_daily = _resample.mean()  # By default downsample using sample average.
ds_daily = ds_daily.assign({
    "tasmin": _resample.min()["tas"],
    "tasmax": _resample.max()["tas"],
    "sfcWindmax": _resample.max()["sfcWind"],
    "tp": _resample.sum()["tp"]
})
ds_daily["tasmin"].attrs["long_name"] = "Minimum "  + ds_daily["tasmin"].attrs["long_name"]
ds_daily["tasmax"].attrs["long_name"] = "Maximum "  + ds_daily["tasmax"].attrs["long_name"]
ds_daily["sfcWindmax"].attrs["long_name"] = "Maximum "  + ds_daily["sfcWindmax"].attrs["long_name"]

# Coordinate time attributes are not persisted during resampling, copy them manually.
ds_daily["time"].attrs.update(ds_hourly["time"].attrs)

display(ds_daily)

<xarray.Dataset> Size: 16MB
Dimensions:     (time: 27028, latitude: 3, longitude: 3)
Coordinates:
  * latitude    (latitude) float32 12B 44.88 45.12 45.38
  * longitude   (longitude) float32 12B 7.375 7.625 7.875
  * time        (time) datetime64[ns] 216kB 1950-01-01 1950-01-02 ... 2023-12-31
Data variables:
    tas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tp          (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    uas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    vas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(11323, 3, 3), meta=np.ndarray>
    sfcWind     (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tasmin      (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tasmax      (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    sfcWindmax  (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Evaluate `pr` variable, which is the quantity actually describing precipitation in CMIP6. This conversion is done on daily data because [xclim does not accept the hour frequency yet](https://github.com/Ouranosinc/xclim/blob/a1a1a25550218330ea36ed428c058275c11579e2/xclim/core/units.py#L409), even if [it is inferred correctly](https://github.com/Ouranosinc/xclim/blob/a1a1a25550218330ea36ed428c058275c11579e2/xclim/core/units.py#L613).

In [12]:
da = xclim.core.units.convert_units_to(
    source=ds_daily["tp"],
    target="kg m-2 s-1",
    context="hydro"
)
da.attrs["long_name"] = "Precipitation"
ds_daily = ds_daily.assign({"pr": da})
display(ds_daily)

<xarray.Dataset> Size: 18MB
Dimensions:     (time: 27028, latitude: 3, longitude: 3)
Coordinates:
  * latitude    (latitude) float32 12B 44.88 45.12 45.38
  * longitude   (longitude) float32 12B 7.375 7.625 7.875
  * time        (time) datetime64[ns] 216kB 1950-01-01 1950-01-02 ... 2023-12-31
Data variables:
    tas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tp          (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    uas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    vas         (time, latitude, longitude) float64 2MB dask.array<chunksize=(11323, 3, 3), meta=np.ndarray>
    sfcWind     (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tasmin      (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    tasmax      (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    sfcWindmax  (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
    pr          (time, latitude, longitude) float64 2MB dask.array<chunksize=(31, 3, 3), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6

Save daily dataset to file for quickier access.

In [13]:
path = dir_out / "reference.nc"
ds_daily.to_netcdf(path)

## Preprocess NEX-GDDP-CMIP6 dataset

Align NEX-GDDP-CMIP6 temporal coordinate to the one of ERA5.

For the midterm discussion only model EC-Earth3 is considered.

In [14]:
dirs = dir_data / "NEX-GDDP"
experiment = ["historical"]
experiment.extend(thesis.scenario)

for e in experiment:
    for m in thesis.models:
        ds = xarray.open_mfdataset(dirs.glob(f"*/day/{m}/{e}/*.nc"), combine_attrs="drop_conflicts")
        
        ds = ds.rename({
            "lat": "latitude",
            "lon": "longitude"
        })
        
        ds = ds.sortby(["latitude", "longitude", "time"])
        
        ds = ds.assign_coords(time=ds["time"].dt.floor("1D"))
        
        ds = ds.sel(
            latitude=slice(thesis.coords[location][0] - offset, thesis.coords[location][0] + offset),
            longitude=slice(thesis.coords[location][1] - offset, thesis.coords[location][1] + offset),
            drop=True
        )
        
        for key in itertools.chain(ds.coords, ds.data_vars):
            ds[key].attrs["long_name"] = ds_daily[key].attrs["long_name"]
        
        path = dir_out / f"{e}_{m}.nc"
        ds.to_netcdf(path)
        
        display(ds)

<xarray.Dataset> Size: 4MB
Dimensions:    (time: 23741, latitude: 3, longitude: 3)
Coordinates:
  * longitude  (longitude) float64 24B 7.375 7.625 7.875
  * latitude   (latitude) float64 24B 44.88 45.12 45.38
  * time       (time) datetime64[ns] 190kB 1950-01-01 1950-01-02 ... 2014-12-31
Data variables:
    pr         (time, latitude, longitude) float32 855kB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    sfcWind    (time, latitude, longitude) float32 855kB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 855kB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmax     (time, latitude, longitude) float32 855kB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmin     (time, latitude, longitude) float32 855kB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
Attributes: (12/21)
    cmip6_source_id:       EC-Earth3
    cmip6_institution_id:  EC-Earth-Consortium
    activity:              NEX-GDDP-CMIP6
    contact:               Dr. Rama Nemani: rama.nemani@nasa.gov, Dr. Bridget...
    Conventions:           CF-1.7
    frequency:             day
    ...                    ...
    disclaimer:            This data is considered provisional and subject to...
    external_variables:    areacella
    geospatial_lat_min:    44.5
    geospatial_lat_max:    45.75
    geospatial_lon_min:    7.0
    geospatial_lon_max:    8.25

<xarray.Dataset> Size: 6MB
Dimensions:    (time: 31411, latitude: 3, longitude: 3)
Coordinates:
  * longitude  (longitude) float64 24B 7.375 7.625 7.875
  * latitude   (latitude) float64 24B 44.88 45.12 45.38
  * time       (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    pr         (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    sfcWind    (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmax     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmin     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
Attributes: (12/22)
    cmip6_source_id:       EC-Earth3
    cmip6_institution_id:  EC-Earth-Consortium
    activity:              NEX-GDDP-CMIP6
    contact:               Dr. Rama Nemani: rama.nemani@nasa.gov, Dr. Bridget...
    Conventions:           CF-1.7
    frequency:             day
    ...                    ...
    external_variables:    areacella
    geospatial_lat_min:    44.5
    geospatial_lat_max:    45.75
    geospatial_lon_min:    7.0
    geospatial_lon_max:    8.25
    doi:                   https://doi.org/10.7917/OFSG3345

<xarray.Dataset> Size: 6MB
Dimensions:    (time: 31411, latitude: 3, longitude: 3)
Coordinates:
  * longitude  (longitude) float64 24B 7.375 7.625 7.875
  * latitude   (latitude) float64 24B 44.88 45.12 45.38
  * time       (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    pr         (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    sfcWind    (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmax     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmin     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
Attributes: (12/21)
    cmip6_source_id:       EC-Earth3
    cmip6_institution_id:  EC-Earth-Consortium
    activity:              NEX-GDDP-CMIP6
    contact:               Dr. Rama Nemani: rama.nemani@nasa.gov, Dr. Bridget...
    Conventions:           CF-1.7
    frequency:             day
    ...                    ...
    disclaimer:            This data is considered provisional and subject to...
    external_variables:    areacella
    geospatial_lat_min:    44.5
    geospatial_lat_max:    45.75
    geospatial_lon_min:    7.0
    geospatial_lon_max:    8.25

<xarray.Dataset> Size: 6MB
Dimensions:    (time: 31411, latitude: 3, longitude: 3)
Coordinates:
  * longitude  (longitude) float64 24B 7.375 7.625 7.875
  * latitude   (latitude) float64 24B 44.88 45.12 45.38
  * time       (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    pr         (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    sfcWind    (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmax     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
    tasmin     (time, latitude, longitude) float32 1MB dask.array<chunksize=(365, 3, 3), meta=np.ndarray>
Attributes: (12/21)
    cmip6_source_id:       EC-Earth3
    cmip6_institution_id:  EC-Earth-Consortium
    activity:              NEX-GDDP-CMIP6
    contact:               Dr. Rama Nemani: rama.nemani@nasa.gov, Dr. Bridget...
    Conventions:           CF-1.7
    frequency:             day
    ...                    ...
    disclaimer:            This data is considered provisional and subject to...
    external_variables:    areacella
    geospatial_lat_min:    44.5
    geospatial_lat_max:    45.75
    geospatial_lon_min:    7.0
    geospatial_lon_max:    8.25